# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [1]:
import pathlib

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsSeqRecDataModule,
    fetch_dataset,
    fetch_metadata,
    bipartite_graph_preprocess_dataset,
)

In [2]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

2025-06-08 17:34:21.916 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:fetch_dataset:39 - Fetching Amazon Reviews 2023 dataset


user_id,parent_asin,rating,timestamp,history
str,str,str,str,str
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07SRWRH5D""","""5.0""","""1587051114941""",""""""
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07DK1H3H5""","""4.0""","""1608186804795""","""B07SRWRH5D"""
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""","""B07MFMFW34""","""5.0""","""1490877431000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B00HUWA45W""","""5.0""","""1427591932000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B0BCHWZX95""","""5.0""","""1577637634017""","""B00HUWA45W"""


In [3]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

Dataset Size
train: 3847041, valid: 344592, test: 363867


The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [4]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(["parent_asin", "title", "categories"])
metadata_df.head(5)

2025-06-08 17:34:24.881 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:fetch_metadata:59 - Fetching Amazon Reviews 2023 metadata


parent_asin,title,categories
str,str,list[str]
"""B000FH0MHO""","""Dash 8-300 Professional Add-On""","[""Video Games"", ""PC"", ""Games""]"
"""B00069EVOG""","""Phantasmagoria: A Puzzle of Fl…","[""Video Games"", ""PC"", ""Games""]"
"""B00Z9TLVK0""","""NBA 2K17 - Early Tip Off Editi…","[""Video Games"", ""PlayStation 4"", ""Games""]"
"""B07SZJZV88""","""Nintendo Selects: The Legend o…","[""Video Games"", ""Legacy Systems"", … ""Games""]"
"""B002WH4ZJG""","""Thrustmaster Elite Fitness Pac…","[""Video Games"", ""Legacy Systems"", … ""Fitness Accessories""]"


In [5]:
print(f"Parent Asin Size: {len(metadata_df)}")

Parent Asin Size: 137269


The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [6]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

2025-06-08 17:34:26.348 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:prepare_data:682 - Loading preprocessed dataset


In [7]:
datamodule.train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,history,history_index,history_category,history_category_index
str,i64,str,i64,str,i64,f64,i64,list[str],list[i64],list[str],list[i64]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07SRWRH5D""",30008,"""Video Games/PlayStation 4/Game…",164,5.0,1587051114941,[],[],[],[]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07DK1H3H5""",27742,"""Video Games/PC/Games""",146,4.0,1608186804795,"[""B07SRWRH5D""]",[30008],"[""Video Games/PlayStation 4/Games""]",[164]
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""",1576453,"""B07MFMFW34""",29081,"""Video Games/PC/Games""",146,5.0,1490877431000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961308,"""B00HUWA45W""",19292,"""Video Games/Xbox One/Accessori…",177,5.0,1427591932000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961308,"""B0BCHWZX95""",33732,"""Video Games/Nintendo Switch/Ac…",121,5.0,1577637634017,"[""B00HUWA45W""]",[19292],"[""Video Games/Xbox One/Accessories""]",[177]


In [8]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

(tensor([2017790, 1921071]),
 tensor([24566, 31019]),
 tensor([[21179,  5833],
         [17200, 20326]]),
 tensor([[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]]))

## Bipartite Graph

In [9]:
(
    train_df,
    val_df,
    test_df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

2025-06-08 17:35:27.278 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:bipartite_graph_preprocess_dataset:872 - Preprocessing the dataset for bipartite graph


In [10]:
train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,num_ratings
str,i64,str,i64,str,i64,f64,i64,u32
"""AELHFSFNZGMUKRQ6Q2MYN367HZUA""",291949,"""B003NSLGW2""",13036,"""Video Games/Legacy Systems/Pla…",82,5.0,1463487981000,1
"""AFQTTZF33CBXX25UDHM4IS7ISSTA""",920300,"""B016XBGWAQ""",22448,"""Video Games/PC""",136,5.0,1482505730000,1
"""AE2PPMOOEKDVEYFWW4EHVXMPKQMA""",11416,"""B001VLFCVE""",11126,"""Video Games/Legacy Systems/Pla…",65,4.0,1292962324000,1
"""AEPDDADHBGVCZI7EGM7Q66BXMQ2Q""",356792,"""B001C6GVI6""",9129,"""Video Games/Legacy Systems/Pla…",65,4.0,1533968227912,1
"""AGPWP7OQVNUCDJDCKSG6KGGZ3L4A""",1442193,"""B0000A2MAQ""",null,"""Video Games/PC""",136,4.0,1431286272000,1
